In [ ]:
import sys
sys.path.append("/path/to/example/autocompute/static")
from g16_env import load_gaussian_env
load_gaussian_env()

In [ ]:
import numpy as np
from rdkit import Chem
from rdkit.Chem import Draw
from more_itertools import chunked
import matplotlib.pyplot as plt
from rdkit.Chem.Draw import rdMolDraw2D
from IPython.display import SVG
import networkx as nx
import pandas as pd
from rdkit.Chem.rdchem import RWMol
from rdkit.Chem import rdmolops
from rdkit.Chem.Draw import IPythonConsole
import seaborn as sns
import re
from rdkit.Chem import AddHs
from rdkit.Chem import AllChem
import os
from openbabel import openbabel, pybel
from rdkit.Chem.rdmolfiles import MolToSmiles
from rdkit.Chem import RemoveHs
import subprocess
import csv
from concurrent.futures import ThreadPoolExecutor, as_completed
from openbabel import openbabel
from openpyxl import Workbook
import time
from openbabel import pybel
import shutil
import glob
from IPython.display import display
from rdkit.Chem import Draw
import random
from rdkit import Chem
from openbabel import openbabel as ob

In [ ]:
import os
os.environ['PATH'] += ':/path/to/example/static/Multiwfn_3.8_dev_bin_Linux'

In [ ]:
IPythonConsole.ipython_useSVG = True

In [ ]:
random_seed = 2 



In [ ]:

df = pd.read_excel('System_random_copolymer.xlsx')


polymer_smiles = {}
polymer_repeating_unit = {}


for index, row in df.iterrows():
    
    if row['is polymer']:
        
        polymer_smiles[row['Name']] = row['SMILES']
        
        polymer_repeating_unit[row['Name']] = row['repeating unit']



if 'copolymer_name' in df.columns:
    
    copolymer_name = df['copolymer_name'].iloc[0]
    print("copolymer_name:")
    print(copolymer_name)
else:
    print("The column 'name_of_polymer' does not exist in the Excel file.")



print("Polymer SMILES Dictionary:")
print(polymer_smiles)
print("\nPolymer Repeating Unit Dictionary:")
print(polymer_repeating_unit)


In [ ]:

df = pd.read_excel('System_random_copolymer.xlsx')


polymer_smiles = {}
polymer_repeating_unit = {}


for index, row in df.iterrows():
    
    if row['is polymer']:
        
        polymer_smiles[row['Name']] = row['SMILES']
        
        polymer_repeating_unit[row['Name']] = row['repeating unit']



if 'copolymer_name' in df.columns:
    
    copolymer_name = df['copolymer_name'].iloc[0]
    print("copolymer_name:")
    print(copolymer_name)
else:
    print("The column 'name_of_polymer' does not exist in the Excel file.")



print("Polymer SMILES Dictionary:")
print(polymer_smiles)
print("\nPolymer Repeating Unit Dictionary:")
print(polymer_repeating_unit)


In [ ]:

def repeat_unit_charge_list(chg_filename):
    
    repeat_unit_charges = []
    with open(chg_filename, 'r') as file:
        for line in file:
            
            repeat_unit_charges.append(float(line.split()[-1]))
    return repeat_unit_charges

In [ ]:

polymer_repeat_unit = {}


polymer_repeat_unit_charges = {}


for polymer_name, smiles in polymer_smiles.items():
    
    mol = Chem.MolFromSmiles(smiles)
    
    mol = Chem.AddHs(mol)
    
    polymer_repeat_unit[polymer_name] = mol
    
    chg_filename = polymer_name + '.chg'
    charges = repeat_unit_charge_list(chg_filename)
    polymer_repeat_unit_charges[polymer_name] = charges


print(f"Molecules Dictionary:{polymer_repeat_unit}")
for name, mol in polymer_repeat_unit.items():
    print(f"{name}: {mol}")


print("\nPolymer Charges Dictionary:")
print(polymer_repeat_unit_charges)

In [ ]:


def create_atom_charges_dicts(polymer_repeat_unit, polymer_repeat_unit_charges, polymer_repeating_unit, random_seed):
    
    random.seed(random_seed) 
    
    polymer_atom_charges_dict = {}
    polymer_atoms_formal_charged_dict = {}
    repeat_unit_count = sum(polymer_repeating_unit.values()) 
    repeat_unit_names = list(polymer_repeating_unit.keys())
    repeat_unit_tracker = {name: 0 for name in repeat_unit_names}

    for i in range(repeat_unit_count):
        
        while True:
            name = random.choice(repeat_unit_names)
            if repeat_unit_tracker[name] < polymer_repeating_unit[name]:
                repeat_unit_tracker[name] += 1
                break

        mol = polymer_repeat_unit[name]
        charges = polymer_repeat_unit_charges[name]
        mol_with_hydrogens = Chem.AddHs(mol)
        
        
        for atom_idx, atom in enumerate(mol_with_hydrogens.GetAtoms()):
            atom_symbol = atom.GetSymbol()
            
            if atom_symbol == '*':
                atom_name = f"*{atom_idx}_{name}_{i+1}"
            else:
                atom_name = f"{atom_symbol}{atom_idx}_{name}_{i+1}"
            
            atom_charge = charges[atom_idx]
            polymer_atom_charges_dict[atom_name] = atom_charge
            
            
            atom_formal_charge = atom.GetFormalCharge()
            if atom_formal_charge != 0:
                polymer_atoms_formal_charged_dict[atom_name] = atom_formal_charge

    return polymer_atom_charges_dict, polymer_atoms_formal_charged_dict

In [ ]:

polymer_repeat_unit = {}
polymer_repeat_unit_charges = {}
for name, smiles in polymer_smiles.items():
    mol = Chem.MolFromSmiles(smiles)
    polymer_repeat_unit[name] = mol
    chg_filename = name + '.chg'
    charges = repeat_unit_charge_list(chg_filename)
    polymer_repeat_unit_charges[name] = charges


polymer_atom_charges_dict, polymer_atoms_formal_charged_dict = create_atom_charges_dicts(
    polymer_repeat_unit, polymer_repeat_unit_charges, polymer_repeating_unit, random_seed
)


print("Polymer Atom Charges Dictionary:")
print(polymer_atom_charges_dict)
print("\nPolymer Atoms Formal Charged Dictionary:")
print(polymer_atoms_formal_charged_dict)

In [ ]:


polymer_end_group_atom_charges_dict = {}


first_virtual_atom_key = None
last_virtual_atom_key = None


for atom_name in polymer_atom_charges_dict.keys():
    if '*' in atom_name:
        if first_virtual_atom_key is None:
            first_virtual_atom_key = atom_name
        last_virtual_atom_key = atom_name


if first_virtual_atom_key:
    polymer_end_group_atom_charges_dict[first_virtual_atom_key] = polymer_atom_charges_dict[first_virtual_atom_key]
if last_virtual_atom_key and last_virtual_atom_key != first_virtual_atom_key:
    polymer_end_group_atom_charges_dict[last_virtual_atom_key] = polymer_atom_charges_dict[last_virtual_atom_key]


print("Polymer End Group Atom Charges Dictionary:")
print(polymer_end_group_atom_charges_dict)

In [ ]:

polymer_del_end_group_atom_charges_dict = {k: v for k, v in polymer_atom_charges_dict.items() if not k.startswith("*") or k in polymer_end_group_atom_charges_dict}
polymer_del_end_group_atom_charges_dict

In [ ]:

final_polymer_atom_charges_dict = {**polymer_del_end_group_atom_charges_dict, **polymer_end_group_atom_charges_dict}


print(final_polymer_atom_charges_dict)

In [ ]:

polymer_charges_list = list(final_polymer_atom_charges_dict.values())
polymer_charges_list

In [ ]:


bond_list = [Chem.rdchem.BondType.UNSPECIFIED,
             Chem.rdchem.BondType.SINGLE, 
             Chem.rdchem.BondType.DOUBLE,
             Chem.rdchem.BondType.TRIPLE, 
             Chem.rdchem.BondType.QUADRUPLE, 
             Chem.rdchem.BondType.QUINTUPLE,
             Chem.rdchem.BondType.HEXTUPLE, 
             Chem.rdchem.BondType.ONEANDAHALF, 
             Chem.rdchem.BondType.TWOANDAHALF,
             Chem.rdchem.BondType.THREEANDAHALF, 
             Chem.rdchem.BondType.FOURANDAHALF, 
             Chem.rdchem.BondType.FIVEANDAHALF,
             Chem.rdchem.BondType.AROMATIC, 
             Chem.rdchem.BondType.IONIC, 
             Chem.rdchem.BondType.HYDROGEN,
             Chem.rdchem.BondType.THREECENTER,
             Chem.rdchem.BondType.DATIVEONE,
             Chem.rdchem.BondType.DATIVE,
             Chem.rdchem.BondType.DATIVEL, 
             Chem.rdchem.BondType.DATIVER,
             Chem.rdchem.BondType.OTHER,
             Chem.rdchem.BondType.ZERO]

In [ ]:


def polymer_all_bonds_info_and_order(bond_list, polymer_repeat_unit, polymer_repeating_unit, random_seed):
    
    random.seed(random_seed) 
    
    repeat_unit_count = sum(polymer_repeating_unit.values()) 
    repeat_unit_names = list(polymer_repeating_unit.keys())
    repeat_unit_tracker = {name: 0 for name in repeat_unit_names}

    
    polymer_bonds_info_with_order = []

    
    polymer_all_bonds_info = []
    
    for i in range(repeat_unit_count):
        
        while True:
            name = random.choice(repeat_unit_names)
            if repeat_unit_tracker[name] < polymer_repeating_unit[name]:
                repeat_unit_tracker[name] += 1
                break

        mol = polymer_repeat_unit[name]
        mol_with_hydrogens = Chem.AddHs(mol)

        
        for bond in mol_with_hydrogens.GetBonds():
            
            begin_idx = bond.GetBeginAtomIdx()
            end_idx = bond.GetEndAtomIdx()
            begin_atom_symbol = mol_with_hydrogens.GetAtomWithIdx(begin_idx).GetSymbol()
            end_atom_symbol = mol_with_hydrogens.GetAtomWithIdx(end_idx).GetSymbol()

            
            begin_atom_name = f"{begin_atom_symbol}{begin_idx}_{name}_{i+1}"
            end_atom_name = f"{end_atom_symbol}{end_idx}_{name}_{i+1}"

            
            bond_order = bond_list.index(bond.GetBondType())

            
            if begin_atom_symbol == '*':
                begin_atom_name = f"*{begin_idx}_{name}_{i+1}"
            if end_atom_symbol == '*':
                end_atom_name = f"*{end_idx}_{name}_{i+1}"

            
            polymer_bonds_info_with_order.append(((begin_atom_name, end_atom_name), bond_order))

            
            polymer_all_bonds_info.append((begin_atom_name, end_atom_name))
    
    return polymer_all_bonds_info, polymer_bonds_info_with_order

In [ ]:
polymer_all_bonds_info, polymer_bonds_info_with_order = polymer_all_bonds_info_and_order(bond_list, polymer_repeat_unit, polymer_repeating_unit, random_seed)

In [ ]:
print(polymer_all_bonds_info)

In [ ]:
print(polymer_bonds_info_with_order)

In [ ]:

del_end_group_polymer_bonds_info_with_order = [
    bond_info for bond_info in polymer_bonds_info_with_order
    if '*' not in bond_info[0][0] and '*' not in bond_info[0][1]
]


print("Deleted End Group Polymer Bonds Info With Order:")
print(del_end_group_polymer_bonds_info_with_order)

In [ ]:

polymer_dummy_atom_bonds_info = [bond for bond in polymer_all_bonds_info if '*' in bond[0] or '*' in bond[1]]
print(polymer_dummy_atom_bonds_info)

In [ ]:

non_virtual_atoms = []
virtual_atom_indices = []
for i, bond in enumerate(polymer_dummy_atom_bonds_info):
    if '*' in bond[0]:
        non_virtual_atoms.append(bond[1])
        virtual_atom_indices.append(i)
    else:
        non_virtual_atoms.append(bond[0])
        virtual_atom_indices.append(i)
print(non_virtual_atoms)
print(virtual_atom_indices)

In [ ]:

polymer_interchain_bonds_info = []

for i in range(1, len(virtual_atom_indices) - 1, 2):
    new_bond = (non_virtual_atoms[i], non_virtual_atoms[i + 1])
    polymer_interchain_bonds_info.append(new_bond)


polymer_interchain_bonds_info.insert(0, polymer_dummy_atom_bonds_info[0])
polymer_interchain_bonds_info.append(polymer_dummy_atom_bonds_info[-1])

In [ ]:
print(polymer_interchain_bonds_info)

In [ ]:

polymer_interchain_bonds_info_with_order = [
    (bond, 1) for bond in polymer_interchain_bonds_info
]


print("Polymer Interchain Bonds Info With Order:")
print(polymer_interchain_bonds_info_with_order)

In [ ]:

del_end_group_polymer_all_bonds_info = [bond for bond in polymer_all_bonds_info if '*' not in bond[0] and '*' not in bond[1]]


final_polymer_connection_list = del_end_group_polymer_all_bonds_info + polymer_interchain_bonds_info


print("Final Polymer Connection List:")
print(final_polymer_connection_list)

In [ ]:

final_polymer_bonds_info_with_order = del_end_group_polymer_bonds_info_with_order + polymer_interchain_bonds_info_with_order


print("Final Polymer Bonds Info with Order:")
for bond_info in final_polymer_bonds_info_with_order:
    print(bond_info)

In [ ]:



node_list = list(final_polymer_atom_charges_dict.keys())
node_list

In [ ]:
def create_index_formal_charge_dict(node_list, atoms_formal_charged_dict):
    
    index_formal_charge_dict = {}

    
    for index, node in enumerate(node_list):
        if node in atoms_formal_charged_dict:
            
            index_formal_charge_dict[index] = atoms_formal_charged_dict[node]

    
    return index_formal_charge_dict

In [ ]:
create_index_formal_charge_dict(node_list, polymer_atoms_formal_charged_dict)

In [ ]:

def create_undirected_graph(node_list, final_polymer_connection_list):
    
    
    G = nx.Graph()

    
    G.add_nodes_from(node_list)

    
    G.add_edges_from(final_polymer_connection_list)

    
    pos = nx.kamada_kawai_layout(G)

    
    plt.figure(figsize=(12, 8))
    nx.draw(G, pos, with_labels=True, node_color='lightblue', edge_color='gray', node_size=200, font_size=8)
    plt.title("Polymer Chain Graph")
    plt.show()

In [ ]:
create_undirected_graph(node_list, final_polymer_connection_list)

In [ ]:

def create_polymer_adjacency_matrix(node_list, final_polymer_bonds_info_with_order):
    
    polymer_adjacency_matrix = np.zeros((len(node_list), len(node_list)))
    
    
    node_index = {node: idx for idx, node in enumerate(node_list)}
    
    
    for bond, order in final_polymer_bonds_info_with_order:
        node1, node2 = bond
        idx1, idx2 = node_index[node1], node_index[node2]
        polymer_adjacency_matrix[idx1][idx2] = order
        polymer_adjacency_matrix[idx2][idx1] = order  
    return polymer_adjacency_matrix

In [ ]:
adjacency_matrix = create_polymer_adjacency_matrix(node_list, final_polymer_bonds_info_with_order)

In [ ]:

def create_heatmap_polymer_adjacency_matrix(adjacency_matrix, node_list, title='Polymer Adjacency Matrix Heatmap', annot=False, fontsize=5, xlabelsize=5, ylabelsize=5):
    
    plt.figure(figsize=(8, 6))
    
    sns.heatmap(adjacency_matrix, annot=annot, cmap='Greys', xticklabels=node_list, yticklabels=node_list, fmt='g')
    
    plt.title(title, fontsize=fontsize)
    
    plt.xticks(fontsize=xlabelsize)
    plt.yticks(fontsize=ylabelsize)
    
    plt.show()

In [ ]:
create_heatmap_polymer_adjacency_matrix(adjacency_matrix, node_list)

In [ ]:

def create_element_node_list(node_list):
    
    element_node_list = []

    
    pattern = re.compile(r"([A-Z][a-z]*)\d*_*\d*")

    
    for node in node_list:
        if node.startswith("*"):  
            element_node_list.append("*")
        else:
            match = pattern.match(node)
            if match:
                
                element_name = match.group(1)
                element_node_list.append(element_name)
            else:
                
                element_node_list.append(node)
    
    return element_node_list

In [ ]:
element_node_list = create_element_node_list(node_list)

In [ ]:
print(element_node_list)

In [ ]:
print(node_list)

In [ ]:
def create_index_formal_charge_dict(node_list, atoms_formal_charged_dict):
    
    index_formal_charge_dict = {}

    
    for index, node in enumerate(node_list):
        if node in atoms_formal_charged_dict:
            
            index_formal_charge_dict[index] = atoms_formal_charged_dict[node]

    
    return index_formal_charge_dict

In [ ]:
index_formal_charge_dict = create_index_formal_charge_dict(node_list, polymer_atoms_formal_charged_dict)

In [ ]:

def graph2mol(atoms, ad_martrix, index_formal_charge_dict):
    
    new_mol = Chem.RWMol()
    
    atom_index = []
    
    
    for atom_number in range(len(atoms)):
        atom = Chem.Atom(atoms[atom_number])  
        molecular_index = new_mol.AddAtom(atom)  
        atom_index.append(molecular_index)  
    
    
    for index_x, row_vector in enumerate(ad_martrix):
        
        for index_y, bond in enumerate(row_vector):
            
            bond = int(bond)
            
            
            if index_y <= index_x:
                continue
            
            if bond == 0:
                continue
            
            else:
                new_mol.AddBond(atom_index[index_x],
                                atom_index[index_y], 
                                bond_list[bond])  
  
    
    for atom_index, formal_charge in index_formal_charge_dict.items():
        
        atom = new_mol.GetAtomWithIdx(atom_index)

        
        
        neighbors = atom.GetNeighbors()
        for neighbor in neighbors:
            
            if neighbor.GetSymbol() == 'H':
                
                editable_mol.RemoveAtom(neighbor.GetIdx())

        
        atom.SetFormalCharge(formal_charge)

    
    updated_polymer_mol = new_mol.GetMol()

    
    Chem.SanitizeMol(new_mol)
    
    
    new_mol = new_mol.GetMol()
    
    
    return new_mol


In [ ]:
polymer_mol = graph2mol(element_node_list, adjacency_matrix, index_formal_charge_dict)

In [ ]:
polymer_mol

In [ ]:


def cap_with_hydrogen(mol):
    mol_with_h = AddHs(mol)

    
    terminal_indices = [atom.GetIdx() for atom in mol_with_h.GetAtoms() if atom.GetSymbol() == '*']
    print("Terminal indices:", terminal_indices)

    
    non_terminal_indices = [atom.GetIdx() for atom in mol_with_h.GetAtoms() if atom.GetSymbol() != '*']
    print("Non-terminal indices:", non_terminal_indices)

    
    rw_mol = RWMol(mol_with_h)

    
    for idx in terminal_indices:
        rw_mol.ReplaceAtom(idx, Chem.Atom(1))  
        rw_mol.GetAtomWithIdx(idx).SetNoImplicit(False)  

    
    rw_mol.UpdatePropertyCache(strict=False)

    
    capped_mol = rw_mol.GetMol()
    
    return capped_mol

In [ ]:

polymer_capped_mol = cap_with_hydrogen(polymer_mol)

In [ ]:
polymer_capped_mol

In [ ]:


mol_with_implicit_hydrogens = Chem.RemoveHs(polymer_capped_mol)


mol_with_implicit_hydrogens

In [ ]:


def calculate_charge_and_multiplicity(mol):
    
    mol = Chem.AddHs(mol)
    
    
    charge = sum(atom.GetFormalCharge() for atom in mol.GetAtoms())
    
    
    multiplicity = sum([atom.GetNumRadicalElectrons() for atom in mol.GetAtoms()]) + 1
    
    return charge, multiplicity

In [ ]:
def create_polymer_Gaussian_inputfile(polymer_capped_mol, polymer_name):
   
    
    
    embed_status = AllChem.EmbedMolecule(polymer_capped_mol, useRandomCoords=True)

    if embed_status == 0:  
        
        optimization_status = AllChem.MMFFOptimizeMolecule(polymer_capped_mol, mmffVariant="MMFF94")
        if optimization_status >= 0:
            
            print("Optimization with MMFF94 successful.")
        else:
            
            print("Optimization with MMFF94 failed.Optimization with UFF")
            AllChem.UFFOptimizeMolecule(polymer_capped_mol)
            
    else:
        
        print("Failed to generate 3D conformation for the molecule.")
    
    
    
    AllChem.EmbedMolecule(polymer_capped_mol, useRandomCoords=True)
    AllChem.UFFOptimizeMolecule(polymer_capped_mol)
    
    
    obConversion = ob.OBConversion()
    
    
    obConversion.SetInAndOutFormats("sdf", "mol2")
    
    
    ob_mol = ob.OBMol()
    
    
    ob_mol = ob.OBMol()
    notatend = obConversion.ReadString(ob_mol, Chem.MolToMolBlock(polymer_capped_mol))
    
    
    obConversion.WriteFile(ob_mol, f"{polymer_name}.mol2")
    
    
    obConversion.SetOutFormat("pdb")
    
    
    obConversion.WriteFile(ob_mol, f"{polymer_name}.pdb")
    
    
    xyz = Chem.MolToXYZBlock(polymer_capped_mol)
    
    
    filename_xyz = f"{polymer_name}.xyz"

    
    with open(filename_xyz, 'w') as file:
        file.write(xyz)


In [ ]:
create_polymer_Gaussian_inputfile(polymer_capped_mol, copolymer_name)

In [ ]:


def clean_mol2_files(copolymer_name):
    
    unity_atom_attr_pattern = re.compile(r'@<TRIPOS>UNITY_ATOM_ATTR')
    bond_pattern = re.compile(r'@<TRIPOS>BOND')

    
    for filename in os.listdir('.'):
        base_filename = os.path.splitext(filename)[0]
        
        if base_filename == copolymer_name and filename.endswith('.mol2'):
            with open(filename, 'r') as file:
                lines = file.readlines()

            
            in_unity_atom_attr_block = False
            
            new_lines = []

            
            for line in lines:
                
                if unity_atom_attr_pattern.search(line):
                    in_unity_atom_attr_block = True  
                    print('find @<TRIPOS>UNITY_ATOM_ATTR')
                    continue  
                
                
                if in_unity_atom_attr_block and bond_pattern.search(line):
                    in_unity_atom_attr_block = False  
                    print('find @<TRIPOS>UNITY_ATOM_ATTR and @<TRIPOS>BOND')

                
                if not in_unity_atom_attr_block:
                    new_lines.append(line)

            
            if in_unity_atom_attr_block or (new_lines != lines):
                with open(filename, 'w') as file:
                    file.writelines(new_lines)


In [ ]:
clean_mol2_files(copolymer_name)

In [ ]:

def polymer_chg(polymer_name, polymer_charges_list):
    
    
    df_charges = pd.DataFrame(polymer_charges_list, columns=['Charge'])

    
    
    df_xyz = pd.read_csv(f"{polymer_name}.xyz", sep='\s+', names=['Element', 'X', 'Y', 'Z'])

    
    df_xyz = df_xyz.drop(df_xyz.index[0])

    
    df_xyz = df_xyz.reset_index(drop=True)

    
    df_combined = pd.concat([df_xyz, df_charges], axis=1)

    
    output_filename = f"{polymer_name}.chg"

    
    df_combined.to_csv(output_filename, sep=' ', index=False, header=False)

In [ ]:
polymer_chg(copolymer_name, polymer_charges_list)

In [ ]:



from typing import Dict, List  
from rdkit import Chem        
import os                     
import json                   
import re                     

def _sanitize_filename(name: str) -> str:
    
    
    return re.sub(r'[\\/:\*\?"<>\|]+', '_', name)

def save_polymer_atom_lists(polymer_repeat_unit: Dict[str, Chem.Mol], output_dir: str = ".") -> Dict[str, str]:
    
    
    if not isinstance(polymer_repeat_unit, dict):
        raise TypeError("Public message removed for release.")

    
    os.makedirs(output_dir, exist_ok=True)

    
    name_to_path: Dict[str, str] = {}

    
    for name, mol in polymer_repeat_unit.items():
        
        if not isinstance(mol, Chem.Mol):
            raise TypeError("Public message removed for release.")

        
        mol_with_hydrogens = Chem.AddHs(mol)

        
        polymer_atom_list: List[str] = []

        
        for atom_idx, atom in enumerate(mol_with_hydrogens.GetAtoms()):
            
            atom_symbol = atom.GetSymbol()

            
            if atom_symbol == '*':
                
                atom_name = f"*{atom_idx}_{name}"
            else:
                
                atom_name = f"{atom_symbol}{atom_idx}_{name}"

            
            polymer_atom_list.append(atom_name)

        
        safe_name = _sanitize_filename(name)
        file_name = f"{safe_name}_atom_list.json"

        
        file_path = os.path.join(output_dir, file_name)

        
        with open(file_path, "w", encoding="utf-8") as f:
            json.dump(polymer_atom_list, f, ensure_ascii=False, indent=2)

        
        name_to_path[name] = os.path.abspath(file_path)

    
    return name_to_path


def load_polymer_atom_list(file_path: str) -> List[str]:
    
    
    if not os.path.isfile(file_path):
        raise FileNotFoundError("Public message removed for release.")

    
    with open(file_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    
    if not isinstance(data, list):
        raise ValueError("Public message removed for release.")

    
    if not all(isinstance(x, str) for x in data):
        raise ValueError("Public message removed for release.")

    
    return data



paths = save_polymer_atom_lists(polymer_repeat_unit, output_dir='.')


with open("polymer_node_list.json", "w", encoding="utf-8") as f:
    json.dump(node_list, f, ensure_ascii=False, indent=2)

In [ ]:
polymer_repeat_unit